# Chapter 8 — Multi-Agent Systems

Splitting Aegis into a team is usually sold as a capability story: specialists outperform
generalists. That is true, and it is the **less important half**.

The important half: three specialists with three toolsets means exactly one agent can
write to the world. Least privilege stops being a policy you write down and becomes the
org chart.

| Agent | Tools | May write? |
|---|---|---|
| Triage | search_logs, get_user_context | no |
| Investigation | + ip_reputation | no |
| Reporting | create_ticket | **yes** |

**Covered:** §8.1.4 the reader is not the actor · §8.2 orchestrator-worker · §8.3 A2A
handoffs · §8.3.3 delegation bounds · §8.4 parallel execution.


## Setup

This lab installs from **one** `requirements.txt`.



In [1]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


repo: /content/aegis


In [2]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.4/695.4 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [3]:
!python tools/check_env.py --chapter 8

dependencies
  ok      langgraph              open-source agent track (StateGraph/END)
  ok      langchain-core         message and tool primitives
  ok      langchain-community    RAGAS dependency — see the pin note
  ok      google-adk             Google Cloud agent track (Agent, Workflow)
  ok      mcp                    tool discovery and hardening (Ch 3, 9, 11)
  ok      openai                 the default real-model tier
  ok      langchain-openai       wires OpenAI into RAGAS
  ok      ragas                  evaluation (Ch 10)
  ok      sacrebleu              required by RAGAS BleuScore
  ok      opentelemetry-sdk      tracing (Ch 10)
  ok      chromadb               vector store (Ch 6)
  ok      rank-bm25              sparse retrieval for hybrid search (Ch 6)
  ok      pytest                 the test suite

critical pin
  ok      langchain-community 0.3.29 (compatible with ragas)

model access
  absent  OPENAI_API_KEY not set
          Offline labs still run: AEGIS_MODEL=mock
  

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [4]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


model tier: mock


## The pipeline, and the envelope that carries it

Work moves between agents in a typed message, not a blob of text. The field doing the
heavy lifting is `trace_id`: it is constant across the whole incident, so three
independent tool-calling agents become **one auditable investigation**.


In [5]:
import sys
sys.path.insert(0, "labs/chapter-08-multi-agent-systems")   # this chapter's source lives beside the notebook
from common import soc
from common.a2a import new_investigation
from common import workers
from common.model import get_model

soc.reset_tickets()
model = get_model()

message = new_investigation(soc.SEED_ALERT, trace_id="inc-4417")
print(f'opening envelope: task={message.task} to={message.to_agent} trace={message.trace_id}')
print()

for stage in (workers.triage, workers.investigate, workers.report):
    message = stage(message, model)
    print(f'handoff after {stage.__name__:12} -> trace={message.trace_id} '
          f'keys={sorted(message.payload)[:3]}')

print()
print("one trace id across triage, investigation and reporting:",
      message.trace_id == "inc-4417")


opening envelope: task=triage_alert to=triage trace=inc-4417

handoff after triage       -> trace=inc-4417 keys=['alert', 'preliminary_severity', 'privileged']
handoff after investigate  -> trace=inc-4417 keys=['alert', 'evidence', 'preliminary_severity']
handoff after report       -> trace=inc-4417 keys=['alert', 'evidence', 'preliminary_severity']

one trace id across triage, investigation and reporting: True


## The reader is not the actor

Suppose Triage is manipulated — by a bug, a bad prompt, or a prompt injection in the
very logs it just read — into opening (or quietly closing) a ticket.

It has the function name. Nothing stops it from *trying*. The toolset boundary is what
stops it from succeeding, and the model's opinion is never consulted.


In [6]:
from common.workers import TRIAGE_TOOLS, INVEST_TOOLS, REPORT_TOOLS

for role, tools in (("triage", TRIAGE_TOOLS),
                    ("investigation", INVEST_TOOLS),
                    ("reporting", REPORT_TOOLS)):
    may_write = "create_ticket" in tools
    print(f'{role:14} may create_ticket? {str(may_write):5} '
          f'{"" if may_write else "<- denied at the toolset boundary"}')

print()
print("Exactly one agent can change the world.")
print("Chapter 2 wrote 'you may not take remediation actions' into a PROMPT.")
print("A manipulated model can talk itself past prose. It cannot talk itself")
print("past a list membership check.")


triage         may create_ticket? False <- denied at the toolset boundary
investigation  may create_ticket? False <- denied at the toolset boundary
reporting      may create_ticket? True  

Exactly one agent can change the world.
Chapter 2 wrote 'you may not take remediation actions' into a PROMPT.
A manipulated model can talk itself past prose. It cannot talk itself
past a list membership check.


##  Bounding delegation

Agents that can hand work to each other can hand it **back**. Triage delegates to
Investigation, Investigation wants triage context and delegates back, and the two of them
bill you for a conversation that never terminates.

The bound belongs in the envelope, not in any single agent's memory. It is the
multi-agent equivalent of Chapter 1's `max_steps`.


In [7]:
from common.coordination import Delegation, MAX_HANDOFFS

chain = Delegation(trace_id="inc-4417")
print(f'max_handoffs = {MAX_HANDOFFS}')
print()

for i in range(7):
    hop = chain.handoff("triage", "investigation", f"re-examine context {i}")
    if not hop["ok"]:
        print(f'  hop {i}: REFUSED - {hop["reason"]} (after {hop["hops"]} hops)')
        break
    print(f'  hop {i}: ok   cycle_detected={hop["cycle_detected"]}')

print()
print("A cycle is not automatically wrong. An UNBOUNDED one is.")


max_handoffs = 5

  hop 0: ok   cycle_detected=False
  hop 1: ok   cycle_detected=True
  hop 2: ok   cycle_detected=True
  hop 3: ok   cycle_detected=True
  hop 4: ok   cycle_detected=True
  hop 5: REFUSED - max_handoffs exceeded (after 5 hops)

A cycle is not automatically wrong. An UNBOUNDED one is.


## Parallel execution: scatter, then gather

Real SOCs fan out: three signals, three investigators, concurrently. That is faster, and
it tests three things at once — the trace must stay single, no branch may write, and the
cost multiplies roughly linearly in branches.

Merging is where the design lives. Two investigators disagreeing is **not an error**; it
is an input to the verdict. A merge that silently picks one has discarded the most
interesting signal in the run.


In [8]:
from common.coordination import fan_out


def investigate_signal(signal: str) -> dict:
    """One investigator, one signal."""
    if "auth" in signal or "privilege" in signal:
        return {"verdict": "confirmed_compromise", "severity": "critical"}
    return {"verdict": "inconclusive", "severity": "low"}


result = fan_out(["failed auth burst", "unusual source ip", "privilege escalation"],
                 investigate_signal, trace_id="inc-4417")

print(f'branches run: {len(result["branches"])}')
for branch in result["branches"]:
    print(f'  {branch["signal"]:24} {branch["verdict"]:22} trace={branch["trace_id"]}')

print()
print("single trace across all branches:", result["single_trace"])
print("any branch wrote to the world:   ", result["any_branch_wrote"])
print()
print("merged:", result["merged"]["verdict"], "| severity:", result["merged"]["severity"])
print("dissent:", result["merged"]["dissent"], result["merged"]["distinct_verdicts"])
print()
print("The most severe verdict won - safe and expensive. The disagreement is")
print("REPORTED rather than averaged away, because it is the signal worth escalating.")


branches run: 3
  failed auth burst        confirmed_compromise   trace=inc-4417
  unusual source ip        inconclusive           trace=inc-4417
  privilege escalation     confirmed_compromise   trace=inc-4417

single trace across all branches: True
any branch wrote to the world:    False

merged: confirmed_compromise | severity: critical
dissent: True ['confirmed_compromise', 'inconclusive']

The most severe verdict won - safe and expensive. The disagreement is
REPORTED rather than averaged away, because it is the signal worth escalating.


### Is it actually parallel?

"Concurrent" is easy to claim and easy to get wrong. Measure it: give each investigator
a deliberate delay and compare wall-clock time against running them in sequence.

Note what this also shows — fan-out trades **latency for cost**. Each branch is a full
investigation, the most expensive stage in the model, and the bill grows roughly linearly
in branches even as the clock time stays flat.


In [9]:
import time
from concurrent.futures import ThreadPoolExecutor

SIGNALS = ["failed auth burst", "unusual source ip", "privilege escalation"]


def slow_investigate(signal: str) -> dict:
    time.sleep(0.3)                      # stand in for a real model call
    return investigate_signal(signal)


start = time.perf_counter()
[slow_investigate(s) for s in SIGNALS]
sequential = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(SIGNALS)) as pool:
    list(pool.map(slow_investigate, SIGNALS))
parallel = time.perf_counter() - start

print(f'sequential: {sequential:.2f}s')
print(f'parallel:   {parallel:.2f}s   ({sequential / parallel:.1f}x faster)')
print()
print(f'cost, however, is unchanged: {len(SIGNALS)} full investigations either way.')
print("fan-out trades LATENCY for COST, roughly linearly in branches.")


sequential: 0.90s
parallel:   0.30s   (3.0x faster)

cost, however, is unchanged: 3 full investigations either way.
fan-out trades LATENCY for COST, roughly linearly in branches.


---

## What you built

A three-agent team with typed handoffs, one trace per incident, a delegation bound, and
concurrent investigation that preserves both the audit thread and the privilege boundary.

- **Multi-agent is a security architecture** before it is a capability story.
- **Least privilege is a list, not a prompt.**
- **Bound delegation** — agents that can hand work back can loop forever.
- **Disagreement is signal.** Report it; do not average it away.

**Next:** Chapter 9 puts a router in front of this team.
